In [41]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv("Airplane_Crashes_and_Fatalities_Since_1908_20190820105639.csv")

states = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN",
    "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA",
    "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC",
    "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN",
    "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI",
    "Wyoming": "WY"
}

state_patterns = {
    state: re.compile(rf"\b({state}|{abbr})\b")
    for state, abbr in states.items()
}

def extract_state_regex(location):
    if pd.isna(location):
        return None
    
    for state, pattern in state_patterns.items():
        if pattern.search(location):
            return state
    
    return None

df["State"] = df["Location"].apply(extract_state_regex)
df.to_csv("Airplane_Crashes_1908_2019_Cleaned.csv")

## Manual Cleaning in Excel ##
# changed entries that had "Georgia" as a country
# grouped duplicate category for Boeing 737 MAX 8

In [ ]:
df = pd.read_csv("Airplane_Crashes_1908_2019_Cleaned.csv")

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # normalize punctuation/spacing
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


causes = {

    "Terrain": [
        r"\bterrain\b",
        r"\bmountain\b",
        r"\bridge\b",
        r"\bhillside\b",
    ],

    "Combat": [
        r"\bshot down\b",
        r"\banti aircraft fire\b",
        r"\benemy fire\b",
        r"\bmissile\b",
        r"\bsurface to air\b",
        r"\bmilitary fire\b"
    ],

    "Mechanical Failure": [
        r"\bengine failure\b",
        r"\bengine failed\b",
        r"\bdual engine failure\b",
        r"\bhydraulic failure\b",
        r"\belectrical failure\b",
        r"\bflight control failure\b",
        r"\bcontrols jammed\b",
        r"\bstructural failure\b",
        r"\bwing separated\b",
        r"\bwing failure\b",
        r"\bpropeller failure\b",
        r"\blanding gear failure\b",
        r"\bfuel exhaustion\b",
        r"\bran out of fuel\b",
        r"\bmechanical failure\b"
    ],

    "Fuel Exhaustion": [
        r"\bran out of fuel\b",
        r"\bfuel exhaustion\b"
    ],

    "Bird": [
        r"\bbird\b",
        r"\bpigeon\b"
    ],

    "Pilot Error": [
        r"\bpilot error\b",
        r"\bincorrect landing\b",
        r"\bwrong runway\b",
        r"\bloss of control\b",
        r"\bunstable approach\b",
        r"\bhard landing\b",
        r"\brunway excursion\b",
        r"\bimproper flare\b",
        r"\bfailed to maintain altitude\b",
        r"\bdescended below\b"
    ],

    "Weather/Visibility": [
        r"\bthunderstorm\b",
        r"\bwindshear\b",
        r"\bstorm\b",
        r"\blightning\b",
        r"\bdowndraft\b",
        r"\bicing\b",
        r"\bairframe icing\b",
        r"\bheavy rain\b",
        r"\bsnowstorm\b",
        r"\bfog\b",
        r"\blow visibility\b",
        r"\bpoor visibility\b",
        r"\bsevere turbulence\b",
        r"\blow overcast\b",
        r"\bweather\b"
    ],

    "Fire/Explosion": [
        r"\bin flight fire\b",
        r"\bexploded\b",
        r"\bexplosion\b",
        r"\bburst into flames\b"
    ]
}


def classify(summary):

    summary = clean_text(summary)

    labels = []

    for category, patterns in causes.items():

        for pattern in patterns:

            if re.search(pattern, summary):
                labels.append(category)
                break

    if not labels:
        labels.append("Other/Unknown")

    return labels

In [166]:
df["Cause Category"] = df["Summary"].apply(classify)

In [167]:
print(df["Cause Category"].value_counts())

Cause Category
[Other/Unknown]                                                       2594
[Weather/Visibility]                                                   680
[Terrain]                                                              430
[Mechanical Failure]                                                   234
[Fire/Explosion]                                                       200
[Terrain, Weather/Visibility]                                          199
[Pilot Error]                                                          140
[Combat]                                                               139
[Pilot Error, Weather/Visibility]                                       74
[Mechanical Failure, Fuel Exhaustion]                                   39
[Weather/Visibility, Fire/Explosion]                                    36
[Terrain, Pilot Error]                                                  30
[Mechanical Failure, Weather/Visibility]                                29
[Mechanica

In [168]:
# remove labels for tiny or unhelpful categories

unwanted_labels = {
    "Other/Unknown",
    "Bird",
    "Fuel Exhaustion"
}

df_train = df[
    df["Cause Category"].apply(
        lambda labels: not any(
            label in unwanted_labels
            for label in labels
        )
    )
].copy()

# text features

vectorizer = TfidfVectorizer(
    ngram_range=(1,3),
    stop_words="english",
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df_train["Summary"]
    .fillna("")
    .apply(clean_text)
)

# multilabel targets

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(
    df_train["Cause Category"]
)

# training/testing the model

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = OneVsRestClassifier(
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000
    )
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# evaluating how the model performed

print(classification_report(
    y_test,
    y_pred,
    target_names=mlb.classes_
))

                    precision    recall  f1-score   support

            Combat       1.00      0.91      0.96        35
    Fire/Explosion       0.92      0.84      0.88        57
Mechanical Failure       0.82      0.93      0.87        68
       Pilot Error       0.80      0.91      0.85        53
           Terrain       0.99      0.89      0.93       158
Weather/Visibility       0.94      0.89      0.92       199

         micro avg       0.92      0.89      0.91       570
         macro avg       0.91      0.89      0.90       570
      weighted avg       0.93      0.89      0.91       570
       samples avg       0.91      0.91      0.90       570



/opt/homebrew/Cellar/jupyterlab/4.4.6/libexec/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [169]:
# get results for what the model learned

feature_names = vectorizer.get_feature_names_out()

for i, class_name in enumerate(mlb.classes_):

    coef = model.estimators_[i].coef_[0]

    top = coef.argsort()[-20:]

    print(f"\nCLASS: {class_name}")

    for idx in reversed(top):
        print(feature_names[idx])


CLASS: Combat
shot
missile
rebels
forces
enemy
british
fighters
anti aircraft
anti
fighter
air
shot british
shot enemy
surface air
claimed
military
claimed shot
plane shot
air missile
aircraft shot

CLASS: Fire/Explosion
exploded
flames
burst flames
burst
explosion
engine
fuel
crashed exploded
flight
bomb
exploded flames
cabin
exploded burned
plane
return
rear
crashed flight
reporting engine
cargo hold
tank

CLASS: Mechanical Failure
failure
engine
engine failure
engine failed
failed
structural failure
structural
wing separated
separated
wing
mechanical failure
mechanical
takeoff
right
electrical failure
right wing
right wing separated
suffered
emergency
experiencing engine failure

CLASS: Pilot Error
pilot error
loss control
descended
loss
control
error
pilot
hard landing
hard
aircraft descended
loss control aircraft
altitude
control aircraft
descended minimum
safe altitude
crew descended
descended mda
mda
aircraft
minimum altitude

CLASS: Terrain
mountain
terrain
crashed mountain
hi

In [170]:
# assign each crash a primary cause

df["Primary Cause"] = df["Cause Category"].apply(
    lambda x: x[0]
)

# reassign causes to crashes with cause "other/unknown"
# when the model is confident about its predictions

unknown_mask = (
    df["Primary Cause"] == "Other/Unknown"
)

X_unknown_text = (
    df.loc[unknown_mask, "Summary"]
    .fillna("")
    .apply(clean_text)
)

X_unknown = vectorizer.transform(X_unknown_text)

probs = model.predict_proba(X_unknown)

best_idx = probs.argmax(axis=1)

best_scores = probs.max(axis=1)

best_labels = [
    mlb.classes_[i]
    for i in best_idx
]

conf_threshold = 0.85

high_conf_mask = best_scores >= conf_threshold

unknown_indices = df.loc[unknown_mask].index

high_conf_indices = unknown_indices[
    high_conf_mask
]

high_conf_labels = np.array(best_labels)[
    high_conf_mask
]

df.loc[
    high_conf_indices,
    "Primary Cause"
] = high_conf_labels

# save confidence scores

df.loc[
    unknown_indices,
    "Prediction Confidence"
] = best_scores

# see reclassified rows

reclassified = df.loc[
    high_conf_indices,
    [
        "Summary",
        "Primary Cause",
        "Prediction Confidence"
    ]
]

print(reclassified.head(20))

                                                Summary       Primary Cause  \
405   Vanished off the coast of Corsica after sendin...  Mechanical Failure   
930   Crashed and burned moments after takeoff. Fail...  Mechanical Failure   
1148      Crashed after the wing failed during takeoff.  Mechanical Failure   
1261  Cargo plane. Loss of lateral control during a ...         Pilot Error   
1324  The cargo plane lost an engine while taking of...  Mechanical Failure   
1800  The plane disintegrated in flight at 18,000 fe...      Fire/Explosion   
2251  Failed to climb on takeoff and crashed into a ...  Mechanical Failure   
2771  Crashed during take off. Failure of the right ...  Mechanical Failure   
2972  Crashed after the left engine flamed out on ta...  Mechanical Failure   
3363   Lost a wing in heavy turbulence at low altitude.  Weather/Visibility   
3500  The sightseeing  plane crashed into a hotel sh...  Mechanical Failure   
4020  Soon after takeoff, the No. 3 engine caught fi

In [171]:
print(df["Primary Cause"].value_counts())

Primary Cause
Other/Unknown         2579
Terrain                720
Weather/Visibility     718
Mechanical Failure     364
Pilot Error            225
Fire/Explosion         201
Combat                 154
Bird                     6
Name: count, dtype: int64


In [172]:
df.loc[
    df["Primary Cause"] == "Other/Unknown",
    "Summary"
].head(50)

1     Eugene Lefebvre was the first pilot to ever be...
3     The first fatal airplane accident in Canada oc...
5     Hydrogen gas which was being vented was sucked...
8            Crashed near the Black Sea, cause unknown.
13                             Caught fire and crashed.
15    Crashed into the sea from an altitude of 3,000...
23    Carl Smith was killed when his mail plane feet...
24    Caught fire in midair. The pilot leaped from t...
25    The mail plane crashed under unknown circumsta...
26     The dirigible, cruising at 1,200 ft. caught f...
27    As the plane was passing over Verona the wings...
28                                                  NaN
31    While on a mail flight to New York the pilot e...
32        Crashed in a field while attemptting to land.
35    After a fire erupted in flight the pilot decid...
36    The aircraft crashed while on approach for unk...
40    The plane crashed during a cargo flight under ...
43    The pilot crashed after attempting to make

In [153]:
df.to_csv("aircraft_crashes_with_causes.csv")